[<img src="../quantumsymmetry_logo.png" alt="QuantumSymmetry" width="450"/>](https://github.com/dariopicozzi/quantumsymmetry)

> **Note:** if you are running this notebook on Google Colab, the next cell will install `quantumsymmetry` and its dependencies:

In [ ]:
%%capture
if 'google.colab' in str(get_ipython()):
    !pip -q install quantumsymmetry

# The complete active space (CAS) approximation

In the previous notebooks we have used `quantumsymmetry` to reduce the number of qubits in the qubit Hamiltonian by exploiting the **exact** Boolean symmetries of the molecule: the point group and the spin parities. The ground-state energy is unchanged by this reduction, as the projection is onto a symmetry sector that the Hamiltonian already preserves.

We can squeeze the qubit count down even further if we are willing to accept a small, controlled approximation. In the **complete active space (CAS)** approach, the molecular orbitals are split into three subspaces:

- the **frozen-core orbitals**, assumed to be doubly occupied;
- the **active orbitals**, whose occupancies are allowed to vary and which capture electron correlation;
- the **virtual orbitals**, assumed to be unoccupied.

The notation $(n, m)$-CAS means that $n$ electrons are distributed in all possible ways across $m$ active spatial orbitals. The frozen-core/virtual choice is an *approximate* $Z$-symmetry: it fixes the occupancy of those orbitals to 1 (frozen-core) or 0 (virtual), exactly as the parity symmetries fix the parity of a set of qubits, and the qubits associated with these orbitals can be removed using the same affine Clifford machinery.

The combined approach is the **symmetry-adapted encoding with complete active space (SAE-CAS)** introduced in the article:

> *Picozzi, D. and Tennyson, J. (manuscript). Symmetry-adapted qubit encoding with complete active space and Bravyi-Kitaev mapping for quantum chemistry on a quantum computer.*

In this notebook we are going to apply SAE-CAS to two illustrative molecules, beryllium hydride (BeH₂) and water (H₂O), in the minimal STO-3G basis. We will see that the qubit savings are significant, and that the ground-state energy in the active space matches the PySCF CASCI reference to numerical precision.

1. [Beryllium hydride (BeH₂) with CAS(2,3)](#BeH2)
2. [Water (H₂O) with CAS(8,6)](#H2O)
3. [How accurate is the active-space approximation?](#accuracy)


<a name="BeH2"></a>
## Beryllium hydride (BeH₂) with CAS(2,3)

Beryllium hydride is a linear molecule (Be in the middle, two H's along the $z$-axis). In the minimal STO-3G basis it has 7 spatial orbitals, or 14 spin-orbitals, and 6 electrons. In tutorial [03_molecular_hamiltonians](03_molecular_hamiltonians.ipynb) we saw that the symmetry-adapted encoding alone reduces the qubit count from 14 down to 9 by exploiting the $D_{2h}$ point group and the spin parities.

The two lowest-energy molecular orbitals, $1a_g$ and $2a_g$, are very close to the doubly-occupied $1s$ orbital of beryllium and the bonding combination of the two hydrogen $1s$ atomic orbitals, and remain essentially fully occupied in the ground state. We can freeze them. Similarly, the highest-energy virtual orbitals (several $\sigma$* and $\pi$* orbitals) remain essentially unoccupied, and we can drop them.

We keep the two $\sigma$-bonding/antibonding orbitals ($1b_{1u}$, $3a_g$) and one $\pi$ orbital ($1b_{2u}$ or $1b_{3u}$, they are degenerate in the linear geometry), which gives a **CAS(2,3)**: 2 electrons in 3 active spatial orbitals. The remaining 4 occupied spin-orbitals are frozen-core and the 8 highest-lying spin-orbitals are virtual.

We pass the active space to `Encoding` with the optional argument `CAS=(n_electrons, n_active_orbitals)`:

In [ ]:
import quantumsymmetry

encoding = quantumsymmetry.Encoding(
    atom = 'Be 0 0 0; H 0 0 1.33; H 0 0 -1.33',
    basis = 'sto-3g',
    CAS = (2, 3),
    verbose = True,
    show_lowest_eigenvalue = True,
)

Reading the verbose report from top to bottom: the molecule has 14 spin-orbitals; the SAE-CAS classifies the lowest 4 as frozen-core, the next 6 as active, and the top 4 as virtual. On top of that, the exact symmetry generators ($P_\uparrow$, $P_\downarrow$, and the relevant reflections in $D_{2h}$) each remove an additional qubit. The overall reduction is

$$14 \;\rightarrow\; 2 \quad \text{qubits}.$$

The qubit-reduced Hamiltonian has 10 Pauli terms on 2 qubits, down from 596 Pauli terms on 9 qubits in the symmetry-adapted encoding without CAS. The ground state of the reduced Hamiltonian matches the FCI/CASCI energy of CAS(2,3) for BeH₂ in STO-3G to numerical precision.

Note that the orbital classification was inferred from PySCF's Hartree-Fock orbital energies. If we want to specify a custom active space (for instance to include or exclude specific orbitals by symmetry), we can pass the `active_mo` argument with a list of 1-based MO indices instead:

In [ ]:
encoding = quantumsymmetry.Encoding(
    atom = 'Be 0 0 0; H 0 0 1.33; H 0 0 -1.33',
    basis = 'sto-3g',
    CAS = (2, 3),
    active_mo = [3, 4, 5],   # 1-based MO indices to use as active
    verbose = False,
)
print('Final number of qubits:', encoding.hamiltonian.many_body_order())

<a name="H2O"></a>
## Water (H₂O) with CAS(8,6)

The water molecule has 10 electrons distributed in 7 spatial orbitals in the minimal STO-3G basis: a deep $1a_1$ orbital (essentially the oxygen $1s$), and 6 valence orbitals. The $1a_1$ orbital is energetically isolated and remains doubly occupied to a very good approximation, which makes it the textbook example of a *chemical frozen core*: we freeze it and let the other 8 electrons distribute across the remaining 6 active orbitals. This is a **CAS(8,6)** with no virtual orbitals.

H₂O is in the $C_{2v}$ point group, which has 2 independent Boolean symmetries on top of the two spin parities.

In [ ]:
encoding = quantumsymmetry.Encoding(
    atom = 'O 0 0 0.119; H 0 0.758 -0.477; H 0 -0.758 -0.477',
    basis = 'sto-3g',
    CAS = (8, 6),
    verbose = True,
    show_lowest_eigenvalue = True,
)

Here the CAS approximation contributes 2 qubit reductions (freezing the oxygen $1s$ spin-up and spin-down spin-orbitals), and the exact symmetries contribute 4 more (the two spin parities and the two non-trivial point-group reflections $\sigma_v(xz)$ and $\sigma_v(yz)$). Overall, the 14 spin-orbital Hamiltonian reduces to **8 qubits**.

By contrast, the symmetry-adapted encoding without CAS for H₂O in STO-3G reduces the system to 10 qubits (see the relevant section in tutorial [03_molecular_hamiltonians](03_molecular_hamiltonians.ipynb)). Freezing the chemically inert oxygen $1s$ buys us 2 additional qubits, while shifting the recovered energy by a fraction of a milli-Hartree at this level of theory.

<a name="accuracy"></a>
## How accurate is the active-space approximation?

Because the CAS projection fixes the occupancy of frozen-core and virtual orbitals, the reduced Hamiltonian is in general not exactly equivalent to the original molecular Hamiltonian: we are choosing a subspace where some orbitals are doubly occupied and others empty, and ignoring the (small) admixture of configurations outside this subspace.

To quantify the error in a controlled way, we can use PySCF's CASCI as the reference, which is the exact diagonalisation of the molecular Hamiltonian projected onto the same active space. The SAE-CAS qubit Hamiltonian's lowest eigenvalue should match the PySCF CASCI energy to numerical precision, because the SAE-CAS construction is provably equivalent to the canonical frozen-core/CAS projection (see Appendix A in the SAE-CAS paper).

Let's verify this for BeH₂ CAS(2,3):

In [ ]:
import numpy as np
from pyscf import gto, scf, mcscf
from openfermion.linalg import qubit_operator_sparse

atom = 'Be 0 0 0; H 0 0 1.33; H 0 0 -1.33'
basis = 'sto-3g'

# Reference CASCI from PySCF
mol = gto.M(atom = atom, basis = basis, symmetry = False, verbose = 0)
mf  = scf.RHF(mol).run(verbose = 0)
cas = mcscf.CASCI(mf, ncas = 3, nelecas = 2).run(verbose = 0)
casci_energy = cas.e_tot

# SAE-CAS qubit Hamiltonian eigenvalue
encoding = quantumsymmetry.Encoding(
    atom = atom, basis = basis, CAS = (2, 3),
)
H = qubit_operator_sparse(encoding.hamiltonian).toarray()
sae_cas_energy = float(np.linalg.eigvalsh(H).min())

print(f'PySCF CASCI energy     : {casci_energy:.10f} Ha')
print(f'SAE-CAS qubit ground   : {sae_cas_energy:.10f} Ha')
print(f'Absolute difference    : {abs(casci_energy - sae_cas_energy):.2e} Ha')

The two energies match to better than 10⁻¹⁰ Hartree — the only remaining difference is floating-point arithmetic. The CAS error compared to FCI is a separate (and physically meaningful) quantity, controlled by the choice of active space, not by the SAE-CAS encoding itself.

<p style="text-align: left"> <a href="05_VQE_circuits.ipynb" />< Previous: Running a variational algorithm with a symmetry-adapted encoding</a> </p>
<p style="text-align: right"> <a href="07_bravyi_kitaev.ipynb" />Next: The Bravyi-Kitaev mapping ></a> </p>